# Machine Learning Models

In [3]:
%%time

%pip install xgboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import gc
import re
import random
from IPython.display import display, HTML
import openpyxl
from sklearn.metrics import (r2_score,mean_absolute_error,mean_squared_error,mean_tweedie_deviance)
from sklearn.model_selection import ParameterGrid, ParameterSampler
from functools import partial

# Pandas DataFrames and Series
pd.set_option("display.float_format",lambda x: f"{x:,.5f}")

# NumPy arrays
np.set_printoptions(suppress=True,precision=6)

# Matplotlib axes
plt.rcParams["axes.formatter.useoffset"] = False
plt.rcParams["axes.formatter.limits"] = (-10, 10)

df = pd.read_csv("crypto_master_dataset_imputed.csv")
list(df.columns)

CPU times: total: 52.2 s
Wall time: 54.5 s


['coin',
 'timestamp',
 'open',
 'high',
 'low',
 'close',
 'ohlcv_volume',
 'momentum_5m',
 'momentum_15m',
 'realized_vol_5m',
 'realized_vol_15m',
 'best_bid',
 'best_ask',
 'spread',
 'spread_pct',
 'bid_depth_top5',
 'ask_depth_top5',
 'bid_depth_top10',
 'ask_depth_top10',
 'order_book_imbalance_top5',
 'order_book_imbalance_top10',
 'open_interest',
 'funding_rate',
 'cumulative_funding',
 'long_account',
 'short_account',
 'long_short_ratio',
 'long_short_skew',
 'liquidation_volume',
 'liquidation_acceleration_5m',
 'liquidation_acceleration_15m',
 'whale_tx_count_100k',
 'whale_tx_count_1m',
 'active_addresses_24h',
 'transaction_volume',
 'social_volume',
 'social_dominance',
 'sentiment_polarity',
 'abnormal_attention',
 'liq_vol_zscore',
 'liq_vol_bucket',
 'liq_vol_flag',
 'long_short_new_observation',
 'oi_momentum_5m',
 'oi_momentum_15m',
 'oi_pct_change_5m',
 'oi_pct_change_15m',
 'oi_zscore_15m',
 'oi_zscore_1h',
 'oi_zscore_3h',
 'price_segment_id',
 'volatility_acce

In [4]:
%%time


# ============================================================
# 1. Configuration
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

HORIZON = 12
SEQUENCE_LENGTH = 60

TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15

TABULAR_LAGS = [12,48]
TARGET_COL = "target_cumulative_liq_volume_next_12row"

model_df = df.copy()

EXCLUDE_COIN = False # Only mark as true when excluding coin during ablation study script!
    
# ============================================================
# 2. Clean and sort
# ============================================================

model_df["timestamp"] = pd.to_datetime(model_df["timestamp"],utc=True,errors="coerce")
model_df["liquidation_volume"] = pd.to_numeric(model_df["liquidation_volume"],errors="coerce")

model_df = (model_df.dropna(subset=["coin","timestamp","price_segment_id","liquidation_volume"])
            .sort_values(["coin","timestamp","price_segment_id"]).reset_index(drop=True))

# ============================================================
# 3. Add time features
# ============================================================

model_df["hour"] = model_df["timestamp"].dt.hour
model_df["minute"] = model_df["timestamp"].dt.minute
model_df["day_of_week"] = model_df["timestamp"].dt.dayofweek

model_df["hour_sin"] = np.sin(2 * np.pi * model_df["hour"] / 24)
model_df["hour_cos"] = np.cos(2 * np.pi * model_df["hour"] / 24)
model_df["minute_sin"] = np.sin(2 * np.pi * model_df["minute"] / 60)
model_df["minute_cos"] = np.cos(2 * np.pi * model_df["minute"] / 60)
model_df["day_sin"] = np.sin(2 * np.pi * model_df["day_of_week"] / 7)
model_df["day_cos"] = np.cos(2 * np.pi * model_df["day_of_week"] / 7)

# ============================================================
# 4. Create the cumulative future target
# ============================================================

# liquidation volume is calculated as the sum from T+1 to T+12 steps
def future_rolling_sum(series, horizon):
    return (series.shift(-1).iloc[::-1].rolling(window=horizon,min_periods=horizon).sum().iloc[::-1])

segment_keys = ["coin","price_segment_id"]

# Grouping by "price segment" prevents the target crossing periods where price data was missing
model_df[TARGET_COL] = (model_df.groupby(["coin","price_segment_id"],sort=False,observed=True)["liquidation_volume"]
                        .transform(lambda series: future_rolling_sum(series,HORIZON)))

# Any rows without a complete future 12-row window are removed
model_df = (model_df.dropna(subset=[TARGET_COL]).reset_index(drop=True))

# here is another check to ensure the target variable is never negative
model_df[TARGET_COL] = (model_df[TARGET_COL].clip(lower=0).astype("float32"))

# ============================================================
# 5. Create naive previous-12-row persistence baseline
# ============================================================

NAIVE_BASELINE_COL = "naive_previous_12row_liq_sum"
# for a given row t, this is taking the sum from t-11 to t
model_df[NAIVE_BASELINE_COL] = (model_df.groupby(segment_keys,sort=False,observed=True)["liquidation_volume"]
                                .transform(lambda series: (series.rolling(window=HORIZON,min_periods=HORIZON).sum())).astype("float32"))

# ============================================================
# 6. Define feature groups
# ============================================================

FEATURE_GROUPS = {
    
    "price_volume": [
        "ohlcv_volume",
        "momentum_5m",
        "momentum_15m",
        "realized_vol_5m",
        "realized_vol_15m",
        "volatility_acceleration",
        "volatility_ratio"
    ],

    "order_book": [
        "spread_pct",
        "total_depth_top5",
        "total_depth_top10",
        "order_book_imbalance_top5",
        "order_book_imbalance_top10",
        "spread_volatility_ratio"
    ],

    "open_interest": [
        "open_interest",
        "oi_momentum_5m",
        "oi_momentum_15m",
        "oi_pct_change_5m",
        "oi_pct_change_15m",
        "oi_zscore_15m",
        "oi_zscore_1h",
        "oi_zscore_3h",
        "oi_acceleration",
        "oi_volatility_interaction"
    ],

    "funding_positioning": [
        "funding_rate",
        "long_short_ratio",
        "long_short_skew"
    ],

    "onchain_activity": [
        "whale_tx_count_100k",
        "whale_tx_count_1m",
        "active_addresses_24h",
        "transaction_volume"
    ],

    "social": [
        "social_volume",
        "social_dominance",
        "sentiment_polarity",
        "abnormal_attention"
    ],

    "time": [
        "hour_sin",
        "hour_cos",
        "minute_sin",
        "minute_cos",
        "day_sin",
        "day_cos"
    ]
}

# ============================================================
# 7. Identify available modelling features
# ============================================================

# Coin identity is handled separately through dummy encoding later.
available_feature_groups = {}
for group_name, group_columns in FEATURE_GROUPS.items():
    available_columns = [column for column in group_columns if column in model_df.columns]
    available_feature_groups[group_name] = (available_columns)


# Flatten all available non-coin feature groups into one list.
# These are the numeric predictors used by both the LSTM and the tabular models.
base_feature_cols = list(dict.fromkeys(column for group_columns in available_feature_groups.values() for column in group_columns))

# ============================================================
# 8. Define variables receiving explicit tabular lags
# ============================================================

preferred_lag_sources = [
    "ohlcv_volume",
    "momentum_5m",
    "momentum_15m",
    "realized_vol_5m",
    "realized_vol_15m",
    "spread_pct",
    "total_depth_top5",
    "total_depth_top10",
    "order_book_imbalance_top5",
    "order_book_imbalance_top10",
    "open_interest",
    "oi_momentum_5m",
    "oi_momentum_15m",
    "oi_pct_change_5m",
    "oi_pct_change_15m",
    "oi_zscore_15m",
    "oi_zscore_1h",
    "oi_zscore_3h",
    "social_volume",
    "social_dominance",
    "sentiment_polarity",
    "abnormal_attention",
    "volatility_acceleration",
    "volatility_ratio",
    "spread_volatility_ratio",
    "oi_acceleration",
    "oi_volatility_interaction"
]

tabular_lag_source_cols = [column for column in preferred_lag_sources if column in model_df.columns]

# ============================================================
# 9. Create lags within coin and price segments
# ============================================================

segment_group = model_df.groupby(segment_keys,sort=False,observed=True)

lag_frames = []
for lag in TABULAR_LAGS:

    lag_frame = (segment_group[tabular_lag_source_cols].shift(lag))

    lag_frame.columns = [f"{column}_lag{lag}"for column in tabular_lag_source_cols]

    lag_frames.append(lag_frame.astype("float32"))

generated_lag_cols = [column for lag_frame in lag_frames for column in lag_frame.columns]

model_df = pd.concat([model_df, *lag_frames],axis=1,copy=False)

del segment_group
del lag_frames
del lag_frame
gc.collect()

# ============================================================
# 10. Add segment-position metadata
# ============================================================

# this is used later as we need enough seequence lengths for lags and LSTM inputs
model_df["segment_position"] = (model_df.groupby(segment_keys,sort=False,observed=True).cumcount().astype("int32"))

# ============================================================
# 11. Split data into train/validation/test
# ============================================================

# -1 represents all rows which will be removed, as their next 12 rows would fall into the next batch of data, therefore resulting in leakage
#  0 = training data
#  1 = validation data
#  2 = test data

model_df["split_id"] = np.int8(-1)

for coin, coin_df in model_df.groupby("coin",sort=False,observed=True):
    
    coin_indices = coin_df.index.to_numpy(dtype=np.int64)
    number_of_rows = len(coin_indices)

    train_end = int(number_of_rows * TRAIN_FRACTION)
    train_stop = max(0,train_end - HORIZON)

    validation_end = int(number_of_rows*(TRAIN_FRACTION+VALIDATION_FRACTION))
    validation_stop = max(train_end,validation_end - HORIZON)

    model_df.loc[coin_indices[:train_stop],"split_id"] = 0
    model_df.loc[coin_indices[train_end:validation_stop],"split_id"] = 1
    model_df.loc[coin_indices[validation_end:],"split_id"] = 2

# ============================================================
# 12. Define identical non-overlapping prediction endpoints
# ============================================================

# A tabular endpoint needs enough history for the longest lag.
# An LSTM endpoint needs SEQUENCE_LENGTH consecutive historical rows.
minimum_history = max(SEQUENCE_LENGTH - 1,max(TABULAR_LAGS))

valid_endpoint = (
    (model_df["segment_position"] >= minimum_history) & # checks it has sufficient length
    (model_df["segment_position"].mod(HORIZON).eq(0))   # prevents the overlap by spacing out by horizon
)

train_idx = model_df.index[(model_df["split_id"] == 0) & valid_endpoint].to_numpy(dtype=np.int64) 
val_idx = model_df.index[(model_df["split_id"] == 1) & valid_endpoint].to_numpy(dtype=np.int64)
test_idx = model_df.index[(model_df["split_id"] == 2) & valid_endpoint].to_numpy(dtype=np.int64)

print("Training:", f"{len(train_idx):,}")
print("Validation:", f"{len(val_idx):,}")
print("Test:", f"{len(test_idx):,}")

# ============================================================
# 13. Fit feature preprocessing using training data only
# ============================================================

# Use the full pre-validation training period to estimate medians, means and standard deviations.
preprocessing_train_idx = model_df.index[model_df["split_id"] == 0].to_numpy(dtype=np.int64)

def fit_feature_statistics(data,columns,fit_indices,):
    retained_columns = []
    statistics = {}

    for column in columns:

        training_values = pd.to_numeric(data.loc[fit_indices,column],errors="coerce").replace([np.inf, -np.inf],np.nan)

        unique_count = (training_values.nunique(dropna=True))

        if unique_count <= 1:
            continue

        median = float(training_values.median())

        filled_values = (training_values.fillna(median))

        mean = float(filled_values.mean())

        standard_deviation = float(filled_values.std(ddof=0))

        if (not np.isfinite(standard_deviation)or standard_deviation == 0):
            continue

        retained_columns.append(column)

        statistics[column] = {"median": median, "mean": mean,"standard_deviation":standard_deviation}

    return retained_columns, statistics

# Base features are used by the LSTM sequence
sequence_numeric_cols, sequence_feature_statistics = (fit_feature_statistics(data=model_df,columns=base_feature_cols,fit_indices=preprocessing_train_idx,))

# Tabular models use base features plus explicit lags
retained_lag_cols, lag_feature_statistics = (fit_feature_statistics(data=model_df,columns=generated_lag_cols,fit_indices=preprocessing_train_idx,))

tabular_numeric_cols = (sequence_numeric_cols + retained_lag_cols)

tabular_feature_statistics = {**sequence_feature_statistics,**lag_feature_statistics}

print("\nSequence numeric predictors:",len(sequence_numeric_cols))

print("Tabular numeric predictors:",len(tabular_numeric_cols))

# ============================================================
# 14. Create common coin dummy variables
# ============================================================

coin_categories = sorted(model_df.loc[preprocessing_train_idx,"coin"].astype(str).unique())

# if we exclude the coin from model features, then we dont create encoded features for them
coin_dummy_categories = ([] if EXCLUDE_COIN else coin_categories[1:])

coin_dummy_names = [f"coin_{category}"for category in coin_dummy_categories]

# Register the encoded coin columns as their own feature group
available_feature_groups["coin"] = (coin_dummy_names.copy())

# ============================================================
# 15. Standardised matrix builder
# ============================================================

def build_standardised_matrix(data,indices,numeric_columns,feature_statistics,dummy_categories):
    
    number_of_rows = len(indices)

    number_of_columns = (len(numeric_columns) + len(dummy_categories))

    X = np.empty((number_of_rows,number_of_columns),dtype=np.float32)

    # Numeric features
    for column_number, column in enumerate(numeric_columns):
        
        values = pd.to_numeric(data.loc[indices,column],errors="coerce").to_numpy(dtype=np.float32,copy=True)

        statistics = (feature_statistics[column])

        invalid_mask = ~np.isfinite(values) # checks if the value is a finite number

        values[invalid_mask] = (statistics["median"]) # if value is not a finite number, then we use the median for imputation
        values = (values - statistics["mean"]) / statistics["standard_deviation"] # calculate z-scores and overwrites median with this
        values = np.clip(values,-10,10) # exclude extreme z-score values
        X[:, column_number] = values # assign values back into dataset

    # Add the coin dummy variables
    selected_coins = (data.loc[indices,"coin"].astype(str).to_numpy())
    for dummy_number, category in enumerate(dummy_categories):
        X[:,len(numeric_columns) + dummy_number] = (selected_coins == category).astype("float32")
        
    return X

sequence_feature_names = (sequence_numeric_cols + coin_dummy_names)

tabular_feature_names = (tabular_numeric_cols + coin_dummy_names)


# ============================================================
# 16. Build common tabular matrices
# ============================================================

X_train_tabular = build_standardised_matrix(
    data=model_df,
    indices=train_idx,
    numeric_columns=tabular_numeric_cols,
    feature_statistics=tabular_feature_statistics,
    dummy_categories=coin_dummy_categories
)

X_val_tabular = build_standardised_matrix(
    data=model_df,
    indices=val_idx,
    numeric_columns=tabular_numeric_cols,
    feature_statistics=tabular_feature_statistics,
    dummy_categories=coin_dummy_categories
)

X_test_tabular = build_standardised_matrix(
    data=model_df,
    indices=test_idx,
    numeric_columns=tabular_numeric_cols,
    feature_statistics=tabular_feature_statistics,
    dummy_categories=coin_dummy_categories
)


# ============================================================
# 17. Build common sequence feature matrix
# ============================================================

all_row_indices = np.arange(
    len(model_df),
    dtype=np.int64
)

X_sequence_all = build_standardised_matrix(
    data=model_df,
    indices=all_row_indices,
    numeric_columns=sequence_numeric_cols,
    feature_statistics=sequence_feature_statistics,
    dummy_categories=coin_dummy_categories
)


# ============================================================
# 18. Create common target arrays
# ============================================================

y_all = model_df[TARGET_COL].to_numpy(dtype=np.float32)

y_train = y_all[train_idx]
y_val = y_all[val_idx]
y_test = y_all[test_idx]

# ============================================================
# 19. Create naive baseline prediction arrays
# ============================================================

naive_predictions_all = (model_df[NAIVE_BASELINE_COL].to_numpy(dtype=np.float32))
naive_train_pred = naive_predictions_all[train_idx]
naive_val_pred = naive_predictions_all[val_idx]
naive_test_pred = naive_predictions_all[test_idx]

# ============================================================
# 20. Final leakage checks
# ============================================================

all_predictor_names = list(dict.fromkeys(sequence_feature_names + tabular_feature_names))

print("\nTarget diagnostics")
print("Training mean:", y_train.mean())
print("Validation mean:", y_val.mean())
print("Test mean:", y_test.mean())
print("Training median:", np.median(y_train))


Training: 208,143
Validation: 44,604
Test: 44,614

Sequence numeric predictors: 40
Tabular numeric predictors: 92

Target diagnostics
Training mean: 37483.086
Validation mean: 25954.371
Test mean: 48937.984
Training median: 103.82607
CPU times: total: 1min 26s
Wall time: 1min 30s


In [5]:
list(model_df.columns)

['coin',
 'timestamp',
 'open',
 'high',
 'low',
 'close',
 'ohlcv_volume',
 'momentum_5m',
 'momentum_15m',
 'realized_vol_5m',
 'realized_vol_15m',
 'best_bid',
 'best_ask',
 'spread',
 'spread_pct',
 'bid_depth_top5',
 'ask_depth_top5',
 'bid_depth_top10',
 'ask_depth_top10',
 'order_book_imbalance_top5',
 'order_book_imbalance_top10',
 'open_interest',
 'funding_rate',
 'cumulative_funding',
 'long_account',
 'short_account',
 'long_short_ratio',
 'long_short_skew',
 'liquidation_volume',
 'liquidation_acceleration_5m',
 'liquidation_acceleration_15m',
 'whale_tx_count_100k',
 'whale_tx_count_1m',
 'active_addresses_24h',
 'transaction_volume',
 'social_volume',
 'social_dominance',
 'sentiment_polarity',
 'abnormal_attention',
 'liq_vol_zscore',
 'liq_vol_bucket',
 'liq_vol_flag',
 'long_short_new_observation',
 'oi_momentum_5m',
 'oi_momentum_15m',
 'oi_pct_change_5m',
 'oi_pct_change_15m',
 'oi_zscore_15m',
 'oi_zscore_1h',
 'oi_zscore_3h',
 'price_segment_id',
 'volatility_acce

### Model 1: XG Boost

In [4]:
%%time

from xgboost import (XGBClassifier,XGBRegressor)

# ============================================================
# 1. Configuration
# ============================================================

XGB_TUNING_TRIALS = 30
XGB_MAX_ESTIMATORS = 2000
XGB_EARLY_STOPPING_ROUNDS = 50

# ============================================================
# 2. Define hurdle targets
# ============================================================

xgb_y_train_positive = (y_train > 0).astype("int8")
xgb_y_val_positive = (y_val > 0).astype("int8")
xgb_positive_train_mask = (y_train > 0)
xgb_positive_val_mask = (y_val > 0)

xgb_negative_count = (xgb_y_train_positive == 0).sum()
xgb_positive_count = (xgb_y_train_positive == 1).sum()
xgb_imbalance_ratio = (xgb_negative_count/ xgb_positive_count)
print("Training positive rate:",xgb_y_train_positive.mean())
print("Validation positive rate:",xgb_y_val_positive.mean())

# ============================================================
# 3. Define hyperparameter grid
# ============================================================

xgb_joint_param_space = {
    "learning_rate": [0.01,0.03,0.05,0.10],
    "max_depth": [3,4,6,8],
    "min_child_weight": [5,10,20,50,100],
    "subsample": [0.60,0.80,1.00],
    "colsample_bytree": [0.60,0.80,1.00],
    "gamma": [0.0,0.1,0.5,1.0],
    "reg_alpha": [0.0,0.01,0.10,1.0],
    "reg_lambda": [0.5,1.0,2.0,5.0,10.0],
    "scale_pos_weight": [1.0,np.sqrt(xgb_imbalance_ratio),xgb_imbalance_ratio / 2,xgb_imbalance_ratio], # Classifier-specific parameter
    "tweedie_variance_power": [1.1,1.3,1.5,1.7,1.9] # Regressor-specific parameter
}

xgb_parameter_trials = list(ParameterSampler(xgb_joint_param_space,n_iter=XGB_TUNING_TRIALS,random_state=SEED))

print("Number of tuning trials:",len(xgb_parameter_trials))

# ============================================================
# 4. tuning results storage objects
# ============================================================

xgb_tuning_results = []

# Selectin is based on best RMSE
xgb_best_val_rmse = np.inf

xgb_best_event_model = None
xgb_best_size_model = None
xgb_best_params = None
xgb_best_val_pred = None

# ============================================================
# 5. Joint random search
# ============================================================

for trial_number, params in enumerate(xgb_parameter_trials,start=1):

    print(f"\nTrial {trial_number} "f"of {XGB_TUNING_TRIALS}")
    print(params)

    # ============================================================
    # A. Train event occurrence classifier
    # ============================================================

    candidate_event_model = XGBClassifier(objective="binary:logistic",
                                          n_estimators=XGB_MAX_ESTIMATORS,learning_rate=params["learning_rate"],
                                          max_depth=params["max_depth"],
                                          min_child_weight=params["min_child_weight"],
                                          subsample=params["subsample"],
                                          colsample_bytree=params["colsample_bytree"],
                                          gamma=params["gamma"],
                                          reg_alpha=params["reg_alpha"],
                                          reg_lambda=params["reg_lambda"],
                                          scale_pos_weight=params["scale_pos_weight"],
                                          tree_method="hist",
                                          eval_metric="aucpr",
                                          early_stopping_rounds=(XGB_EARLY_STOPPING_ROUNDS),
                                          random_state=SEED,
                                          n_jobs=-1)

    candidate_event_model.fit(X_train_tabular,
                              xgb_y_train_positive,
                              eval_set=[(X_val_tabular,xgb_y_val_positive)],
                              verbose=False)

    # ============================================================
    # B. Train positive volume regressor
    # ============================================================

    tweedie_power = params["tweedie_variance_power"]

    candidate_size_model = XGBRegressor(objective="reg:tweedie",
                                        tweedie_variance_power=(tweedie_power),
                                        n_estimators=XGB_MAX_ESTIMATORS,
                                        learning_rate=params["learning_rate"],
                                        max_depth=params["max_depth"],
                                        min_child_weight=params["min_child_weight"],
                                        subsample=params["subsample"],
                                        colsample_bytree=params["colsample_bytree"],
                                        gamma=params["gamma"],
                                        reg_alpha=params["reg_alpha"],
                                        reg_lambda=params["reg_lambda"],
                                        tree_method="hist",
                                        eval_metric=(f"tweedie-nloglik@"f"{tweedie_power}"),
                                        early_stopping_rounds=(XGB_EARLY_STOPPING_ROUNDS),
                                        random_state=SEED,
                                        n_jobs=-1)


    candidate_size_model.fit(X_train_tabular[xgb_positive_train_mask],
                            y_train[xgb_positive_train_mask],
                            eval_set=[(X_val_tabular[xgb_positive_val_mask],y_val[xgb_positive_val_mask])],
                            verbose=False)

    # ============================================================
    # C. Validation prediction
    # ============================================================

    candidate_val_event_probability = (candidate_event_model.predict_proba(X_val_tabular)[:, 1])
    candidate_val_conditional_amount = np.clip(candidate_size_model.predict(X_val_tabular ),0,None )
    candidate_val_pred = (candidate_val_event_probability * candidate_val_conditional_amount)
    candidate_val_pred = np.asarray(candidate_val_pred,dtype=np.float64).reshape(-1)

    # ============================================================
    # D. Calculate evaluation metrics
    # ============================================================

    candidate_val_r2 = r2_score(y_val,candidate_val_pred)
    candidate_val_mae = mean_absolute_error(y_val,candidate_val_pred)
    candidate_val_rmse = np.sqrt(mean_squared_error(y_val,candidate_val_pred))

    trial_result = {"trial": trial_number,
                    "validation_rmse": candidate_val_rmse,
                    "validation_mae": candidate_val_mae,
                    "validation_r2": candidate_val_r2,
                    "event_best_iteration": (candidate_event_model.best_iteration),
                    "size_best_iteration": (candidate_size_model.best_iteration),
                    **params}
            
    xgb_tuning_results.append(trial_result)

    print(f"Validation RMSE: "f"{candidate_val_rmse:.6f}")
    print(f"Validation MAE: "f"{candidate_val_mae:.6f}")
    print(f"Validation R²: "f"{candidate_val_r2:.6f}")

    # ============================================================
    # E. Keep track of best model based on RMSE
    # ============================================================

    if candidate_val_rmse < xgb_best_val_rmse:

        if xgb_best_event_model is not None: del xgb_best_event_model
        if xgb_best_size_model is not None: del xgb_best_size_model

        xgb_best_val_rmse = candidate_val_rmse
        xgb_best_event_model = (candidate_event_model)
        xgb_best_size_model = (candidate_size_model)
        xgb_best_params = params.copy()
        xgb_best_val_pred = (candidate_val_pred.copy())

        print("New best model found.")

    else:

        del candidate_event_model
        del candidate_size_model

    gc.collect()

# ============================================================
# 6. Create tuning results table
# ============================================================

xgb_tuning_results_df = pd.DataFrame(xgb_tuning_results).sort_values("validation_rmse",ascending=True).reset_index(drop=True)

print("\nTop tuning results:")

display(xgb_tuning_results_df.head(10))

print("\nBest validation RMSE:",xgb_best_val_rmse)
print("\nBest parameters:")
print(xgb_best_params)


# ============================================================
# 7. Name final selected models
# ============================================================

xgb_event_model = (xgb_best_event_model)
xgb_size_model = (xgb_best_size_model)
xgb_val_pred = (xgb_best_val_pred)

# ============================================================
# 8. Generate untouched test predictions
# ============================================================

xgb_test_event_probability = (xgb_event_model.predict_proba(X_test_tabular)[:, 1])
xgb_test_conditional_amount = np.clip(xgb_size_model.predict(X_test_tabular),0,None)
xgb_test_pred = (xgb_test_event_probability * xgb_test_conditional_amount)
xgb_test_pred = np.asarray(xgb_test_pred,dtype=np.float64).reshape(-1)

# ============================================================
# 9. Final validation and test results
# ============================================================

xgb_final_val_rmse = np.sqrt(mean_squared_error(y_val,xgb_val_pred))
xgb_final_val_mae = mean_absolute_error(y_val,xgb_val_pred)
xgb_final_val_r2 = r2_score(y_val,xgb_val_pred)
xgb_final_test_rmse = np.sqrt(mean_squared_error(y_test,xgb_test_pred))
xgb_final_test_mae = mean_absolute_error(y_test,xgb_test_pred)
xgb_final_test_r2 = r2_score(y_test,xgb_test_pred)

print("\nFinal selected XGBoost model")
print(f"Validation RMSE: "f"{xgb_final_val_rmse:.6f}")
print(f"Validation MAE: "f"{xgb_final_val_mae:.6f}")
print(f"Validation R²: "f"{xgb_final_val_r2:.6f}")
print(f"\nTest RMSE: "f"{xgb_final_test_rmse:.6f}")
print(f"Test MAE: "f"{xgb_final_test_mae:.6f}")
print(f"Test R²: "f"{xgb_final_test_r2:.6f}")


Training positive rate: 0.5571650259677241
Validation positive rate: 0.4828042328042328
Class imbalance ratio: 0.7948003794084677
Number of tuning trials: 30

Trial 1 of 30
{'tweedie_variance_power': 1.7, 'subsample': 0.8, 'scale_pos_weight': np.float64(0.39740018970423385), 'reg_lambda': 2.0, 'reg_alpha': 0.1, 'min_child_weight': 10, 'max_depth': 3, 'learning_rate': 0.03, 'gamma': 0.1, 'colsample_bytree': 0.6}
Validation RMSE: 166007.097654
Validation MAE: 28078.067354
Validation R²: 0.092953
New best model found.

Trial 2 of 30
{'tweedie_variance_power': 1.1, 'subsample': 1.0, 'scale_pos_weight': np.float64(0.7948003794084677), 'reg_lambda': 0.5, 'reg_alpha': 0.01, 'min_child_weight': 100, 'max_depth': 8, 'learning_rate': 0.1, 'gamma': 0.5, 'colsample_bytree': 0.8}
Validation RMSE: 167053.396129
Validation MAE: 29716.380979
Validation R²: 0.081483

Trial 3 of 30
{'tweedie_variance_power': 1.5, 'subsample': 0.8, 'scale_pos_weight': np.float64(0.7948003794084677), 'reg_lambda': 5.0, 'r

,trial,validation_rmse,validation_mae,validation_r2,event_best_iteration,size_best_iteration,tweedie_variance_power,subsample,scale_pos_weight,reg_lambda,reg_alpha,min_child_weight,max_depth,learning_rate,gamma,colsample_bytree
0,7,"165,443.40254","28,281.84073",0.09910,326,124,1.50000,0.60000,1.00000,1.00000,0.10000,100,8,0.03000,1.00000,1.00000
1,18,"165,503.19286","28,614.27526",0.09845,409,132,1.90000,0.80000,1.00000,1.00000,0.10000,20,6,0.03000,1.00000,0.60000
2,8,"165,505.74539","29,015.34511",0.09842,572,140,1.50000,1.00000,0.79480,5.00000,0.01000,100,6,0.03000,0.10000,0.60000
3,17,"165,514.48420","28,577.88899",0.09833,716,440,1.30000,0.60000,1.00000,0.50000,0.00000,10,8,0.01000,0.00000,1.00000
4,12,"165,520.32734","29,837.66538",0.09826,161,79,1.70000,0.60000,1.00000,0.50000,0.00000,10,4,0.10000,0.10000,0.60000
5,5,"165,601.74951","28,580.12842",0.09738,998,366,1.70000,0.60000,0.79480,2.00000,1.00000,10,6,0.01000,0.10000,0.60000
6,30,"165,685.16028","28,591.07453",0.09647,118,34,1.90000,0.80000,0.79480,0.50000,0.10000,5,6,0.10000,0.00000,0.60000
7,14,"165,728.26862","29,103.68962",0.09600,568,216,1.10000,0.80000,0.79480,10.00000,0.00000,100,6,0.03000,0.00000,0.60000
8,19,"165,784.23424","26,689.13629",0.09539,955,298,1.70000,1.00000,0.39740,0.50000,1.00000,100,8,0.01000,0.00000,1.00000
9,9,"165,841.67313","29,096.11697",0.09476,490,212,1.10000,0.80000,0.79480,10.00000,0.00000,50,6,0.03000,0.50000,1.00000



Best validation RMSE: 165443.40253672152

Best parameters:
{'tweedie_variance_power': 1.5, 'subsample': 0.6, 'scale_pos_weight': 1.0, 'reg_lambda': 1.0, 'reg_alpha': 0.1, 'min_child_weight': 100, 'max_depth': 8, 'learning_rate': 0.03, 'gamma': 1.0, 'colsample_bytree': 1.0}

Final selected XGBoost model
Validation RMSE: 165443.402537
Validation MAE: 28281.840725
Validation R²: 0.099103

Test RMSE: 484288.138532
Test MAE: 50304.559764
Test R²: 0.034868
CPU times: total: 4h 36min 37s
Wall time: 42min 24s


### Model 2: Tweedie Regression

In [5]:
%%time

from sklearn.linear_model import TweedieRegressor

# ============================================================
# 1. Configuration
# ============================================================

TWEDIE_MAX_ITER = 500
TWEDIE_TOL = 1e-5

# ============================================================
# 2. Hyperparameter grid
# ============================================================

tweedie_param_grid = {
    "power": [1.1,1.3,1.5,1.7,1.9], # Compound Poisson-Gamma range
    "alpha": [0.0,0.0001,0.001,0.01,0.1,1.0,10.0]} # L2 regularisation strength

tweedie_parameter_trials = list(ParameterGrid(tweedie_param_grid))

print("Number of tuning trials:",len(tweedie_parameter_trials))

# ============================================================
# 3. Storage for tuning results
# ============================================================

tweedie_tuning_results = []

# Selectionis based on RMSE
tweedie_best_val_rmse = np.inf
tweedie_best_model = None
tweedie_best_params = None
tweedie_best_val_pred = None

# ============================================================
# 4. Exhaustive grid search
# ============================================================

for trial_number, params in enumerate(tweedie_parameter_trials,start=1):

    print(f"\nTrial {trial_number} "f"of {len(tweedie_parameter_trials)}")
    print(params)

    # --------------------------------------------------------
    # 4A. Train candidate model
    # --------------------------------------------------------

    candidate_model = TweedieRegressor(power=params["power"],
                                       alpha=params["alpha"],
                                       link="log",
                                       solver="newton-cholesky",
                                       max_iter=TWEDIE_MAX_ITER,
                                       tol=TWEDIE_TOL,
                                       fit_intercept=True,
                                       verbose=0)

    try:
        candidate_model.fit(X_train_tabular,y_train.astype(np.float64))

        # ----------------------------------------------------
        # 4B. Validation predictions
        # ----------------------------------------------------

        candidate_val_pred = np.clip(candidate_model.predict(X_val_tabular),0,None)
        candidate_val_pred = np.asarray(candidate_val_pred,dtype=np.float64).reshape(-1)

        # ----------------------------------------------------
        # 4C. Calculate validation metrics
        # ----------------------------------------------------

        candidate_val_rmse = np.sqrt(mean_squared_error(y_val,candidate_val_pred))
        candidate_val_mae = mean_absolute_error(y_val,candidate_val_pred)
        candidate_val_r2 = r2_score(y_val,candidate_val_pred)
        candidate_val_tweedie_deviance = (mean_tweedie_deviance(y_val,candidate_val_pred,power=params["power"]))

        trial_result = {"trial": trial_number,
                        "validation_rmse": candidate_val_rmse,
                        "validation_mae": candidate_val_mae,
                        "validation_r2": candidate_val_r2,
                        "validation_tweedie_deviance": (candidate_val_tweedie_deviance),
                        "iterations_used": (candidate_model.n_iter_),
                        **params}

        tweedie_tuning_results.append(trial_result)

        print(f"Validation RMSE: "f"{candidate_val_rmse:.6f}")
        print(f"Validation MAE: "f"{candidate_val_mae:.6f}")
        print(f"Validation R²: "f"{candidate_val_r2:.6f}")
        print(f"Iterations: "f"{candidate_model.n_iter_}")

        # ----------------------------------------------------
        # 4D. Retain best model by validation RMSE
        # ----------------------------------------------------

        if candidate_val_rmse < tweedie_best_val_rmse:

            if tweedie_best_model is not None:
                del tweedie_best_model

            tweedie_best_val_rmse = (candidate_val_rmse)
            tweedie_best_model = (candidate_model)
            tweedie_best_params = (params.copy())
            tweedie_best_val_pred = (candidate_val_pred.copy())
            print("New best model found.")

        else:
            del candidate_model

    gc.collect()


# ============================================================
# 5. Create tuning-results table
# ============================================================

tweedie_tuning_results_df = pd.DataFrame(tweedie_tuning_results).sort_values("validation_rmse",ascending=True).reset_index(drop=True)

print("\nTop Tweedie tuning results:")
display(tweedie_tuning_results_df.head(10))
print("\nBest validation RMSE:",tweedie_best_val_rmse)
print("\nBest parameters:")
print(tweedie_best_params)

# ============================================================
# 6. Name final selected model
# ============================================================

tweedie_model = (tweedie_best_model)
tweedie_val_pred = (tweedie_best_val_pred)

# ============================================================
# 7. Generate untouched test predictions
# ============================================================

tweedie_test_pred = np.clip(tweedie_model.predict(X_test_tabular),0,None)
tweedie_test_pred = np.asarray(tweedie_test_pred,dtype=np.float64).reshape(-1)

# ============================================================
# 8. Final validation metrics
# ============================================================

tweedie_final_val_rmse = np.sqrt(mean_squared_error(y_val,tweedie_val_pred))
tweedie_final_val_mae = mean_absolute_error(y_val,tweedie_val_pred)
tweedie_final_val_r2 = r2_score(y_val,tweedie_val_pred)

# ============================================================
# 9. Final test metrics
# ============================================================

tweedie_final_test_rmse = np.sqrt(mean_squared_error(y_test,tweedie_test_pred))
tweedie_final_test_mae = mean_absolute_error(y_test,tweedie_test_pred)
tweedie_final_test_r2 = r2_score(y_test,tweedie_test_pred)

print("\nFinal selected Tweedie Regression")
print(f"Validation RMSE: "f"{tweedie_final_val_rmse:.6f}")
print(f"Validation MAE: "f"{tweedie_final_val_mae:.6f}")
print(f"Validation R²: "f"{tweedie_final_val_r2:.6f}")
print(f"\nTest RMSE: "f"{tweedie_final_test_rmse:.6f}")
print(f"Test MAE: "f"{tweedie_final_test_mae:.6f}")
print(f"Test R²: "f"{tweedie_final_test_r2:.6f}"
     )

Number of tuning trials: 35

Trial 1 of 35
{'alpha': 0.0, 'power': 1.1}
Validation RMSE: 171795.693944
Validation MAE: 32296.858647
Validation R²: 0.028594
Iterations: 10
New best model found.

Trial 2 of 35
{'alpha': 0.0, 'power': 1.3}
Validation RMSE: 172178.637892
Validation MAE: 31950.270669
Validation R²: 0.024258
Iterations: 9

Trial 3 of 35
{'alpha': 0.0, 'power': 1.5}
Validation RMSE: 175670.000404
Validation MAE: 31966.511755
Validation R²: -0.015714
Iterations: 8

Trial 4 of 35
{'alpha': 0.0, 'power': 1.7}
Validation RMSE: 194306.510981
Validation MAE: 33041.413599
Validation R²: -0.242656
Iterations: 7

Trial 5 of 35
{'alpha': 0.0, 'power': 1.9}
Validation RMSE: 371486.959608
Validation MAE: 38641.704241
Validation R²: -3.542171
Iterations: 9

Trial 6 of 35
{'alpha': 0.0001, 'power': 1.1}
Validation RMSE: 171795.693253
Validation MAE: 32296.853959
Validation R²: 0.028594
Iterations: 10
New best model found.

Trial 7 of 35
{'alpha': 0.0001, 'power': 1.3}
Validation RMSE: 1721

,trial,validation_rmse,validation_mae,validation_r2,validation_tweedie_deviance,iterations_used,alpha,power
0,33,"169,631.69671","27,770.83100",0.05291,425.93676,6,10.00000,1.50000
1,34,"170,218.40660","27,231.55146",0.04635,86.12418,6,10.00000,1.70000
2,32,"170,227.15477","30,163.86623",0.04625,"2,749.29863",6,10.00000,1.30000
3,31,"171,324.51973","31,889.56360",0.03391,"23,533.33048",8,10.00000,1.10000
4,30,"171,565.83202","27,053.11401",0.03119,30.09786,7,1.00000,1.90000
5,35,"171,656.56458","29,305.22169",0.03017,31.59634,5,10.00000,1.90000
6,26,"171,699.86510","32,226.17128",0.02968,"23,552.49117",9,1.00000,1.10000
7,27,"171,707.45181","31,563.53972",0.02959,"2,696.82455",6,1.00000,1.30000
8,21,"171,786.99840","32,289.49976",0.02869,"23,575.07598",9,0.10000,1.10000
9,16,"171,795.39089","32,296.27405",0.02860,"23,611.58291",9,0.01000,1.10000



Best validation RMSE: 169631.6967120074

Best parameters:
{'alpha': 10.0, 'power': 1.5}

Final selected Tweedie Regression
Validation RMSE: 169631.696712
Validation MAE: 27770.830997
Validation R²: 0.052912

Test RMSE: 487591.053723
Test MAE: 51136.165402
Test R²: 0.021658
CPU times: total: 20min 53s
Wall time: 12min 1s


### Model 3: LSTM

In [6]:
%%time

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import (Input,LSTM,Dense,Dropout,LayerNormalization,SpatialDropout1D)
from tensorflow.keras.callbacks import (EarlyStopping,ReduceLROnPlateau)
from tensorflow.keras.optimizers import Adam

# Tweedie deviance loss for a given power
def tweedie_deviance_raw(y_true,y_pred,power):
    epsilon = tf.keras.backend.epsilon()
    safe_true = tf.maximum(tf.cast(y_true, tf.float32),0.0)
    safe_prediction = tf.maximum(tf.cast(y_pred, tf.float32),epsilon)
    p = tf.cast(power, tf.float32)
    term_1 = (tf.pow(safe_true, 2.0 - p) / ((1.0 - p) * (2.0 - p)))
    term_2 = (safe_true * tf.pow(safe_prediction, 1.0 - p) / (1.0 - p))
    term_3 = (tf.pow(safe_prediction, 2.0 - p) / (2.0 - p))
    deviance = 2.0 * (term_1 - term_2 + term_3)
    return tf.reduce_mean(deviance)

# ============================================================
# Define LSTM target scaling value
# ============================================================

lstm_positive_training_targets = np.asarray(y_train[y_train > 0],dtype=np.float64)
LSTM_AMOUNT_SCALE = np.quantile(lstm_positive_training_targets,0.95)
if (not np.isfinite(LSTM_AMOUNT_SCALE) or LSTM_AMOUNT_SCALE <= 0): LSTM_AMOUNT_SCALE = 1.0
print("LSTM_AMOUNT_SCALE:",LSTM_AMOUNT_SCALE)

# ============================================================
# Builds a tf.data.Dataset via from_generator
# ============================================================

def build_liquidation_dataset(
    feature_matrix,
    targets,
    endpoint_indices,
    sequence_length,
    number_of_features,
    amount_scale,
    batch_size=512,
    shuffle=False):

    targets = np.asarray(targets)
    endpoint_indices = np.asarray(endpoint_indices, dtype=np.int64)
    sequence_length = int(sequence_length)
    amount_scale = float(amount_scale)
    batch_size = int(batch_size)
    offsets = np.arange(sequence_length - 1,-1,-1,dtype=np.int64)
    
    def generator():
        order = np.arange(len(endpoint_indices))
        if shuffle: np.random.shuffle(order)
        number_of_batches = int(np.ceil(len(order) / batch_size))
        for batch_number in range(number_of_batches):
            start = batch_number * batch_size
            stop = min(start + batch_size, len(order))
            selected_order = order[start:stop]
            batch_endpoints = endpoint_indices[selected_order]
            sequence_indices = (batch_endpoints[:, None] - offsets[None, :])
            X_batch = feature_matrix[sequence_indices].astype("float32")
            y_raw = targets[batch_endpoints].astype("float32")
            event_target = (y_raw > 0).astype("float32").reshape(-1, 1)
            amount_target = ( y_raw / amount_scale).astype("float32").reshape(-1, 1)
            yield X_batch, {"event_output": event_target,"amount_output": amount_target}

    output_signature = (tf.TensorSpec(shape=(None, sequence_length, number_of_features),dtype=tf.float32),
        {"event_output": tf.TensorSpec(shape=(None, 1), dtype=tf.float32),"amount_output": tf.TensorSpec(shape=(None, 1), dtype=tf.float32)})

    dataset = tf.data.Dataset.from_generator(generator,output_signature=output_signature)

    return dataset.prefetch(tf.data.AUTOTUNE)

# ============================================================
# 1. Configuration
# ============================================================

LSTM_TUNING_TRIALS = 15
LSTM_TUNING_EPOCHS = 15
LSTM_BEST_WEIGHTS_PATH = ("best_lstm_tuning.weights.h5")
number_of_sequence_features = (X_sequence_all.shape[1])

# ============================================================
# 2. Hyperparameter search space
# ============================================================

lstm_param_space = {"batch_size": [256,512,1024],
                    "lstm_units_1": [32,64,128],
                    "lstm_units_2": [16,32,64],
                    "dense_units_1": [32,64,128],
                    "dense_units_2": [16,32,64],
                    "spatial_dropout": [0.0,0.10,0.20],
                    "lstm_dropout": [0.10,0.20,0.30],
                    "dense_dropout": [0.10,0.25,0.40],
                    "learning_rate": [0.0001,0.0003,0.001,0.003],
                    "clipnorm": [0.5,1.0,2.0],
                    "amount_loss_weight": [0.5,1.0,2.0,5.0],
                    "tweedie_power": [1.1,1.3,1.5,1.7,1.9]}

lstm_parameter_trials = list(ParameterSampler(lstm_param_space,
                                              n_iter=LSTM_TUNING_TRIALS,
                                              random_state=SEED))

print("Number of LSTM tuning trials:",len(lstm_parameter_trials))

# ============================================================
# 3. Compile the model
# ============================================================

def build_lstm_hurdle_model(params):

    sequence_input = Input(shape=(SEQUENCE_LENGTH,number_of_sequence_features),name="sequence_input")
    x = SpatialDropout1D(rate=params["spatial_dropout"])(sequence_input)
    x = LSTM(units=params["lstm_units_1"],return_sequences=True,name="lstm_1")(x)
    x = Dropout(rate=params["lstm_dropout"])(x)
    x = LSTM(units=params["lstm_units_2"],return_sequences=False,name="lstm_2")(x)
    x = LayerNormalization()(x)
    x = Dense(units=params["dense_units_1"],activation="relu")(x)
    x = Dropout(rate=params["dense_dropout"])(x)
    x = Dense(units=params["dense_units_2"],activation="relu")(x)
    event_output = Dense(units=1,activation="sigmoid",name="event_output")(x)
    amount_output = Dense(units=1,activation="softplus",name="amount_output")(x)
    model = Model(inputs=sequence_input,outputs={"event_output": event_output,"amount_output": amount_output})
    amount_loss_fn = partial(tweedie_deviance_raw,power=params["tweedie_power"])
    amount_loss_fn.__name__ = "tweedie_deviance_raw"  # Keras expects a __name__ attribute

    model.compile(optimizer=Adam(learning_rate=params["learning_rate"],clipnorm=params["clipnorm"]),
        loss={"event_output":"binary_crossentropy","amount_output":amount_loss_fn},
        loss_weights={"event_output": 1.0,"amount_output":params["amount_loss_weight"]},
        metrics={"event_output": [tf.keras.metrics.AUC(curve="PR",name="pr_auc")]})

    return model

# ============================================================
# 4. Prediction function
# ============================================================

def get_lstm_hurdle_prediction(model,sequence,amount_scale,verbose=0):

    outputs = model.predict(sequence,verbose=verbose)
    event_probability = np.asarray(outputs["event_output"],dtype=np.float64).reshape(-1)
    conditional_amount = np.asarray(outputs["amount_output"],dtype=np.float64).reshape(-1)
    conditional_amount = np.clip(conditional_amount * amount_scale,0,None)
    prediction = (event_probability * conditional_amount)
    return np.asarray(prediction,dtype=np.float64).reshape(-1)

# ============================================================
# 5. Tuning results for storage
# ============================================================

lstm_tuning_results = []
lstm_best_val_rmse = np.inf
lstm_best_params = None
lstm_best_val_pred = None
lstm_best_epochs_used = None

# ============================================================
# 6. Random hyperparameter search
# ============================================================

for trial_number, params in enumerate(lstm_parameter_trials,start=1):
    print(f"\nTrial {trial_number} "f"of {LSTM_TUNING_TRIALS}")
    print(params)

    # ============================================================
    # A. Reset TensorFlow state
    # ============================================================

    tf.keras.backend.clear_session()
    random.seed(SEED + trial_number)
    np.random.seed(SEED + trial_number)
    tf.random.set_seed(SEED + trial_number)

    # ============================================================
    # B. Create datasets using candidate batch size
    # ============================================================

    candidate_train_sequence = build_liquidation_dataset(feature_matrix=X_sequence_all,
                                                         targets=y_all,
                                                         endpoint_indices=train_idx,
                                                         sequence_length=SEQUENCE_LENGTH,
                                                         number_of_features=number_of_sequence_features,
                                                         amount_scale=LSTM_AMOUNT_SCALE,
                                                         batch_size=params["batch_size"],
                                                         shuffle=True)
                                                 
    candidate_validation_sequence = build_liquidation_dataset(feature_matrix=X_sequence_all,
                                                              targets=y_all,
                                                              endpoint_indices=val_idx,
                                                              sequence_length=SEQUENCE_LENGTH,
                                                              number_of_features=number_of_sequence_features,
                                                              amount_scale=LSTM_AMOUNT_SCALE,
                                                              batch_size=params["batch_size"],
                                                              shuffle=False)

    # ============================================================
    # C. Build model
    # ============================================================

    candidate_model = (build_lstm_hurdle_model(params))

    # ============================================================
    # D. Train model
    # ============================================================

    candidate_callbacks = [EarlyStopping(monitor="val_loss",mode="min",patience=3,restore_best_weights=True,verbose=1),
                           ReduceLROnPlateau(monitor="val_loss",mode="min",factor=0.5,patience=2,min_lr=1e-5,verbose=1)]

    try:

        candidate_history = (candidate_model.fit(candidate_train_sequence,validation_data=(candidate_validation_sequence),
                epochs=LSTM_TUNING_EPOCHS,callbacks=candidate_callbacks,verbose=1))

    # ============================================================
    # E. Obtain the Validation data predictions
    # ============================================================

        candidate_val_pred = (get_lstm_hurdle_prediction(model=candidate_model,sequence=(candidate_validation_sequence),
                amount_scale=LSTM_AMOUNT_SCALE,verbose=0))

    # ============================================================
    # F. Obtain the Validation data metrics
    # ============================================================

        candidate_val_rmse = np.sqrt(mean_squared_error(y_val,candidate_val_pred))
        candidate_val_mae = (mean_absolute_error(y_val,candidate_val_pred))
        candidate_val_r2 = r2_score(y_val,candidate_val_pred)
        candidate_epochs_used = len(candidate_history.history["loss"])
        candidate_best_val_loss = np.min(candidate_history.history["val_loss"])

        trial_result = {"trial": trial_number,
                        "validation_rmse":candidate_val_rmse,
                        "validation_mae":candidate_val_mae,
                        "validation_r2":candidate_val_r2,
                        "best_validation_loss":candidate_best_val_loss,
                        "epochs_used":candidate_epochs_used,
                        **params}

        lstm_tuning_results.append(trial_result)

        print(f"Validation RMSE: "f"{candidate_val_rmse:.6f}")
        print(f"Validation MAE: "f"{candidate_val_mae:.6f}")
        print(f"Validation R²: "f"{candidate_val_r2:.6f}")

    # ============================================================
    # 6G. Retain best model by validation RMSE
    # ============================================================

        if candidate_val_rmse < lstm_best_val_rmse:
            lstm_best_val_rmse = (candidate_val_rmse)
            lstm_best_params = (params.copy())
            lstm_best_val_pred = (candidate_val_pred.copy())
            lstm_best_epochs_used = (candidate_epochs_used)
            candidate_model.save_weights(LSTM_BEST_WEIGHTS_PATH)
            print("New best LSTM model found.")

    del candidate_model
    del candidate_train_sequence
    del candidate_validation_sequence

    gc.collect()

# ============================================================
# 7. Results table from tuning
# ============================================================

lstm_tuning_results_df = pd.DataFrame(lstm_tuning_results).sort_values("validation_rmse",ascending=True).reset_index(drop=True)

print("\nTop LSTM tuning results:")
display(lstm_tuning_results_df.head(10))
print("\nBest validation RMSE:",lstm_best_val_rmse)
print("\nBest parameters:")
print(lstm_best_params)

# ============================================================
# 8. Rebuild selected model and load its weights
# ============================================================

tf.keras.backend.clear_session()
lstm_model = build_lstm_hurdle_model(lstm_best_params)
lstm_model.load_weights(LSTM_BEST_WEIGHTS_PATH)

# ============================================================
# 9. Recreate selected validation and test datasets
# ============================================================

validation_sequence = build_liquidation_dataset(feature_matrix=X_sequence_all,
                                                targets=y_all,
                                                endpoint_indices=val_idx,
                                                sequence_length=SEQUENCE_LENGTH,
                                                number_of_features=number_of_sequence_features,
                                                amount_scale=LSTM_AMOUNT_SCALE,
                                                batch_size=lstm_best_params["batch_size"],
                                                shuffle=False)

test_sequence = build_liquidation_dataset(feature_matrix=X_sequence_all,
                                          targets=y_all,
                                          endpoint_indices=test_idx,
                                          sequence_length=SEQUENCE_LENGTH,
                                          number_of_features=number_of_sequence_features,
                                          amount_scale=LSTM_AMOUNT_SCALE,
                                          batch_size=lstm_best_params["batch_size"],
                                          shuffle=False)

# ============================================================
# 10. Final validation predictions
# ============================================================

lstm_val_pred = get_lstm_hurdle_prediction(model=lstm_model,sequence=validation_sequence,amount_scale=LSTM_AMOUNT_SCALE,verbose=1)

# ============================================================
# 11. Untouched test predictions
# ============================================================

lstm_test_pred = get_lstm_hurdle_prediction(model=lstm_model,sequence=test_sequence,amount_scale=LSTM_AMOUNT_SCALE,verbose=1)

# ============================================================
# 12. Final metrics
# ============================================================

lstm_final_val_rmse = np.sqrt(mean_squared_error(y_val,lstm_val_pred))
lstm_final_val_mae = mean_absolute_error(y_val,lstm_val_pred)
lstm_final_val_r2 = r2_score(y_val,lstm_val_pred)
lstm_final_test_rmse = np.sqrt(mean_squared_error(y_test,lstm_test_pred))
lstm_final_test_mae = mean_absolute_error(y_test,lstm_test_pred)
lstm_final_test_r2 = r2_score(y_test,lstm_test_pred)

print("\nFinal selected LSTM hurdle model")
print(f"Validation RMSE: "f"{lstm_final_val_rmse:.6f}")
print(f"Validation MAE: "f"{lstm_final_val_mae:.6f}")
print(f"Validation R²: "f"{lstm_final_val_r2:.6f}")
print(f"\nTest RMSE: "f"{lstm_final_test_rmse:.6f}")
print(f"Test MAE: "f"{lstm_final_test_mae:.6f}")
print(f"Test R²: "f"{lstm_final_test_r2:.6f}")

C:\Users\admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\google\protobuf\runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
C:\Users\admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Pyt

LSTM_AMOUNT_SCALE: 278722.4046875
Number of LSTM tuning trials: 15

Trial 1 of 15
{'tweedie_power': 1.7, 'spatial_dropout': 0.1, 'lstm_units_2': 16, 'lstm_units_1': 64, 'lstm_dropout': 0.1, 'learning_rate': 0.0003, 'dense_units_2': 16, 'dense_units_1': 64, 'dense_dropout': 0.4, 'clipnorm': 2.0, 'batch_size': 256, 'amount_loss_weight': 0.5}

Epoch 1/15
    814/Unknown 116s 133ms/step - amount_output_loss: 2.7440 - event_output_loss: 0.6390 - event_output_pr_auc: 0.7314 - loss: 2.0110

C:\Users\admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


814/814 ━━━━━━━━━━━━━━━━━━━━ 129s 150ms/step - amount_output_loss: 2.2920 - event_output_loss: 0.6040 - event_output_pr_auc: 0.7720 - loss: 1.7528 - val_amount_output_loss: 1.7536 - val_event_output_loss: 0.5317 - val_event_output_pr_auc: 0.8160 - val_loss: 1.4159 - learning_rate: 3.0000e-04
Epoch 2/15
814/814 ━━━━━━━━━━━━━━━━━━━━ 124s 152ms/step - amount_output_loss: 2.0112 - event_output_loss: 0.5718 - event_output_pr_auc: 0.8025 - loss: 1.5790 - val_amount_output_loss: 1.7158 - val_event_output_loss: 0.5227 - val_event_output_pr_auc: 0.8228 - val_loss: 1.3879 - learning_rate: 3.0000e-04
Epoch 3/15
814/814 ━━━━━━━━━━━━━━━━━━━━ 109s 134ms/step - amount_output_loss: 1.9575 - event_output_loss: 0.5659 - event_output_pr_auc: 0.8095 - loss: 1.5468 - val_amount_output_loss: 1.7351 - val_event_output_loss: 0.5285 - val_event_output_pr_auc: 0.8239 - val_loss: 1.4035 - learning_rate: 3.0000e-04
Epoch 4/15
813/814 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step - amount_output_loss: 1.9388 - event_output_l

C:\Users\admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


...


Best validation RMSE: 166661.34275619176

Best parameters:
{'tweedie_power': 1.5, 'spatial_dropout': 0.2, 'lstm_units_2': 64, 'lstm_units_1': 32, 'lstm_dropout': 0.1, 'learning_rate': 0.003, 'dense_units_2': 16, 'dense_units_1': 64, 'dense_dropout': 0.1, 'clipnorm': 0.5, 'batch_size': 512, 'amount_loss_weight': 0.5}


C:\Users\admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 34 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


88/88 ━━━━━━━━━━━━━━━━━━━━ 8s 86ms/step


C:\Users\admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


88/88 ━━━━━━━━━━━━━━━━━━━━ 8s 89ms/step

Final selected LSTM hurdle model
Validation RMSE: 166661.342756
Validation MAE: 28067.721400
Validation R²: 0.085790

Test RMSE: 484876.189514
Test MAE: 50781.574193
Test R²: 0.032523
CPU times: total: 23h 51min 49s
Wall time: 5h 11min 32s
